# 00 · Bootstrap

Run this first. It proves the machine is ready, and fails loudly and specifically
instead of leaving you guessing three chapters later.

> **You'll learn**
> - Why agents run as *processes* here, and the notebook only drives them
> - How to tell a registered node from a live one
> - How to read a run's execution graph — the visual every chapter turns on

### The one idea to carry forward

The notebook is a **cockpit**, not a runtime. Agents run as separate processes
registered with a control plane; this notebook drives them over HTTP. Keep that
split in mind — every chapter builds on it.

In [1]:
import os, sys, json, time, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent / "lib"))

OK, BAD, WARN = "\u2713", "\u2717", "!"
results = {}

def report(name, passed, detail=""):
    results[name] = bool(passed)
    print(f"{OK if passed else BAD} {name:<22} {detail}")
    return passed

def note(msg):
    print(f"{WARN} {msg}")

## 1 · Python 3.13

`agentfield` requires `>=3.10,<3.14`. On 3.14 the install resolves to nothing.

In [2]:
v = sys.version_info
report("python 3.13", v[:2] == (3, 13), f"{v.major}.{v.minor}.{v.micro}  ({pathlib.Path(sys.executable).parent.parent.name}/)")
if v[:2] != (3, 13):
    note("Wrong interpreter. Run `make setup`, then use the .venv kernel.")

✓ python 3.13            3.13.5  (.venv/)


## 2 · The SDK imports

In [3]:
try:
    import agentfield
    report("agentfield import", agentfield.__version__ == "0.1.132", agentfield.__version__)
except Exception as e:
    report("agentfield import", False, repr(e))

✓ agentfield import      0.1.132


## 3 · Configuration

`AI_MODEL` needs the `openrouter/` prefix. Without it the call still succeeds — it
just quietly goes to a different provider. *([why](../docs/gotchas.md))*

In [4]:
REPO = pathlib.Path.cwd().parent
env_path = REPO / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, val = line.partition("=")
            os.environ.setdefault(k.strip(), val.strip())

SERVER = os.environ.get("AGENTFIELD_SERVER", "http://localhost:8080")
MODEL  = os.environ.get("AI_MODEL", "(unset)")
KEY    = os.environ.get("OPENROUTER_API_KEY", "")

report(".env present", env_path.exists(), str(env_path.relative_to(REPO)))
report("AGENTFIELD_SERVER", bool(SERVER), SERVER)
report("AI_MODEL prefixed", MODEL.startswith("openrouter/"), MODEL)
report("OPENROUTER_API_KEY", KEY.startswith("sk-or-") , "set" if KEY.startswith("sk-or-") else "MISSING or placeholder")
None


✓ .env present           .env
✓ AGENTFIELD_SERVER      http://localhost:8080
✓ AI_MODEL prefixed      openrouter/deepseek/deepseek-v4-flash
✓ OPENROUTER_API_KEY     set


## 4 · The control plane

One process, holding every registered agent and every recorded run.

In [5]:
import httpx

try:
    r = httpx.get(f"{SERVER}/health", timeout=5)
    h = r.json()
    report("control plane", r.status_code == 200 and h.get("status") == "healthy",
           f"{h.get('status')}  v{h.get('version')}")
except Exception as e:
    report("control plane", False, f"{e!r}")
    note("Start it with `make up` (or `af server`).")

✓ control plane          healthy  v1.0.0


## 5 · Registered, or actually alive?

Two different questions. Registrations outlive the process that made them, so a node
whose process died still shows up here looking healthy. Only its own `/health`
answers honestly.

In [6]:
import dag

try:
    caps = dag.nodes(SERVER)
    report("capabilities", True, f"{len(caps)} agent(s) registered")
    live = []
    for c in caps:
        aid = c.get("agent_id")
        if c.get("health_status") == "active" and dag.node_alive(aid, SERVER):
            live.append(c)
    print()
    dag.print_nodes(SERVER)
    print()
    report("at least one LIVE node", bool(live),
           ", ".join(c["agent_id"] for c in live) or "none answering their own /health")
except Exception as e:
    report("capabilities", False, repr(e))
    caps, live = [], []

✓ capabilities           11 agent(s) registered

   (10 unrelated agent(s) on this control plane, not shown)
   1 agent(s) registered:
     - blast-radius [active] 24 reasoner(s) @ http://127.0.0.1:8002
   NOTE: registrations outlive the process. health_status can lie;
         dag.node_alive(<agent_id>) pings the node itself.

✓ at least one LIVE node blast-radius, brsmoke


## 6 · One dispatch

Async, because it returns the `run_id` the DAG needs.

Two ids come back and they are not interchangeable: `execution_id` belongs to
`/api/v1/executions/{id}`, `run_id` to `/api/v1/agentic/run/{id}`. `lib/dag.py`
takes either.

In [7]:
PREFERRED = "blast-radius"   # this repo's own node, once node/main.py is running
run_id = None

target_agent = next((c for c in live if c.get("agent_id") == PREFERRED), None) \
            or (live[0] if live else None)

if not target_agent:
    note("No live node to dispatch to. Start one with `make up`, then re-run.")
    note("Chapters 01+ bring up node/main.py; bootstrap tolerates its absence.")
    report("dispatch", False, "skipped - no live node")
else:
    aid = target_agent["agent_id"]
    reasoner = (target_agent.get("reasoners") or [{}])[0].get("id")
    # invocation_target uses a colon (agent:reasoner) but the execute endpoint
    # requires a dot (agent.reasoner). Passing the colon form returns HTTP 400.
    tgt = f"{aid}.{reasoner}"
    try:
        r = httpx.post(f"{SERVER}/api/v1/execute/async/{tgt}", json={"input": {}}, timeout=60)
        body = r.json()
        run_id = body.get("run_id") or body.get("workflow_id")
        # Any answer that came back FROM the node proves the round trip, even a
        # schema rejection: the control plane reached it and it replied.
        report("dispatch round-trip", r.status_code < 500, f"{tgt} -> HTTP {r.status_code}")
        if run_id:
            print(f"  run_id={run_id}")
        else:
            print(f"  no run_id in response: {json.dumps(body)[:200]}")
    except Exception as e:
        report("dispatch round-trip", False, repr(e))

✓ dispatch round-trip    blast-radius.r01_diagnose -> HTTP 202
  run_id=run_20260820_125414_lipukbyj


## 7 · The DAG

Every call an agent makes is recorded as an edge. `lib/dag.py` fetches the run and
renders it as mermaid, natively — no CDN, no JavaScript.

This picture is the argument of the whole repo. By chapter 04 it will be identical on
every run; by chapter 06, different every time.

In [8]:
target_run = run_id
if not target_run:
    try:
        recent = dag.recent_runs(SERVER)
        if recent:
            target_run = recent[0]["run_id"]
            note(f"No run of our own; rendering the most recent run on the control plane: {target_run}")
    except Exception as e:
        note(f"could not list recent runs: {e!r}")

out = None
if target_run:
    try:
        time.sleep(1.5)   # let the first executions land
        d = dag.fetch_run(target_run, SERVER)
        s = dag.stats(d)
        report("DAG fetched", True,
               f"{s['executions']} executions, depth {s['max_depth']}, fan-out {s['max_fanout']}")
        out = dag.render(target_run, server=SERVER, title=f"bootstrap run {target_run}")
    except Exception as e:
        report("DAG fetched", False, repr(e))
else:
    report("DAG fetched", False, "no run available to render")
out

✓ DAG fetched            1 executions, depth 1, fan-out 0


**bootstrap run run_20260820_125414_lipukbyj** — 1 executions · depth 1 · max fan-out 0 · 1 agent(s)

```mermaid
flowchart TD
  n0["r01_diagnose<br/><small>✗ failed · 0.0s</small>"]
  class n0 bad;
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

## 8 · Summary

In [9]:
print("bootstrap summary")
print("-" * 46)
for k, v in results.items():
    print(f"  {OK if v else BAD} {k}")
required = ["python 3.13", "agentfield import", "control plane"]
missing = [k for k in required if not results.get(k)]
print("-" * 46)
if missing:
    print(f"{BAD} NOT READY - fix: {', '.join(missing)}")
else:
    print(f"{OK} core environment is ready.")
    soft = [k for k, v in results.items() if not v]
    if soft:
        print(f"{WARN} non-blocking: {', '.join(soft)}")
        print("  (these need a running node - `make up` - and are expected to be")
        print("   red until node/main.py exists.)")

bootstrap summary
----------------------------------------------
  ✓ python 3.13
  ✓ agentfield import
  ✓ .env present
  ✓ AGENTFIELD_SERVER
  ✓ AI_MODEL prefixed
  ✓ OPENROUTER_API_KEY
  ✓ control plane
  ✓ capabilities
  ✓ at least one LIVE node
  ✓ dispatch round-trip
  ✓ DAG fetched
----------------------------------------------
✓ core environment is ready.


---

## What you learned

- Agents are **processes**; the notebook is a client, not a host
- **Registered ≠ alive** — check the node's own `/health`
- A run's DAG comes from `/api/v1/agentic/run/{run_id}` and renders inline

**Next:** `01_one_shot.ipynb` — one call, one schema, and a model that tells you when
it isn't sure.